# 01 · Quality flags and repeated printings (steps 2–3)

Shows how often each quality flag fires per decade, and random flagged regions next to their scanned clipping, raw OCR and post-corrected text. Judge whether each flag is right.

Needs `python -m hisrag.ingest`, `python -m hisrag.normalize text` and `python -m hisrag.normalize dedup` to have run.

| Flag | Meaning |
|---|---|
| `pc_repetition` | post-correction looped or duplicated text (a 4-word sequence ≥4×, at least 2× as often as in the OCR) |
| `pc_expanded` | post-correction is >1.3× as long as the OCR (duplicated or added text) |
| `pc_unsupported` | <60 % of the corrected text's character trigrams occur in the OCR: reconstructed from noise, possibly invented |
| `too_short` | under 30 characters |
| `death_register` | an entry of the Vienna death register (name, occupation, age, address, cause of death), not an ad |

In [ ]:
import html
from IPython.display import HTML, display
from hisrag.data import query

FLAGS = ["pc_repetition", "pc_expanded", "pc_unsupported", "too_short", "death_register"]
query("SELECT decade, count(*) AS regions, "
      + ", ".join(f"sum(flag_{f}::int) AS {f}" for f in FLAGS)
      + ", round(100 * avg((n_flags > 0)::int), 1) AS pct_any FROM ad_text JOIN ads USING (ad_id) GROUP BY 1 ORDER BY 1")

In [ ]:
def show(where: str, n: int = 5, seed: int = 0):
    """Random regions matching a SQL condition on ads/ad_text, with clipping, OCR and correction side by side."""
    rows = query(f"SELECT t.*, a.label, a.text, a.text_ocr, a.iiif_link FROM ad_text t JOIN ads a USING (ad_id) WHERE {where} "
                 f"ORDER BY hash(ad_id || '{seed}') LIMIT {n}")
    for r in rows.itertuples():
        flags = [f for f in FLAGS if getattr(r, f"flag_{f}")]
        esc = lambda s: html.escape(s or "").replace("\n", "<br>")
        display(HTML(f"""
        <div style="border-top:2px solid #888;margin-top:14px;padding-top:6px">
          <b>{r.ad_id}</b> · {r.year} · {r.label} · flags: <b>{", ".join(flags) or "none"}</b>
          · support {r.ocr_support:.2f} · length ratio {r.len_ratio:.2f} · max repeat {r.repeat_max}
          <div><img src="{r.iiif_link}" style="max-width:100%;max-height:420px;margin:6px 0"></div>
          <div style="display:flex;gap:16px;font-size:12px">
            <div style="flex:1"><i>OCR</i><br>{esc(r.text_ocr)}</div>
            <div style="flex:1"><i>post-corrected</i><br>{esc(r.text)}</div>
          </div>
        </div>"""))

## `pc_unsupported`

In [ ]:
show("flag_pc_unsupported")

## `pc_repetition`

In [ ]:
show("flag_pc_repetition")

## `pc_expanded`

In [ ]:
show("flag_pc_expanded")

## `death_register`

In [ ]:
show("flag_death_register")

## `too_short`

In [ ]:
show("flag_too_short")

## Unflagged, for comparison
Change `seed` to see other samples.

In [ ]:
show("n_flags = 0", seed=1)

# Repeated printings (step 3)

Regions judged to be the same ad printed on different days (within 60 days of each other, same newspaper). Every cluster has one canonical member, the one with the fewest quality flags and the best OCR support. Check that all members of a cluster really are the same ad. The most likely error is two notices from the same template (different school or court) merged into one.

In [ ]:
query("""SELECT dup_cluster_size AS printings, count(*) AS clusters, round(avg(run_days), 1) AS avg_run_days
         FROM ad_dups WHERE is_canonical GROUP BY 1 ORDER BY 1""")

In [ ]:
def show_cluster(where: str = "dup_cluster_size BETWEEN 2 AND 6", seed: int = 0, max_members: int = 6):
    """One random cluster matching a condition on ad_dups: all members with date, clipping and text."""
    cid = query(f"SELECT dup_cluster_id FROM ad_dups WHERE is_canonical AND {where} "
                f"ORDER BY hash(ad_id || '{seed}') LIMIT 1").iloc[0, 0]
    rows = query(f"""SELECT d.ad_id, d.is_canonical, d.dup_cluster_size, a.date, a.label, a.iiif_link, a.text
                     FROM ad_dups d JOIN ads a USING (ad_id) WHERE d.dup_cluster_id = '{cid}'
                     ORDER BY a.date LIMIT {max_members}""")
    parts = [f"<h4>{cid} · {rows.dup_cluster_size.iloc[0]} printings</h4>"]
    for r in rows.itertuples():
        mark = " <b>(canonical)</b>" if r.is_canonical else ""
        parts.append(f"""<div style="border-top:1px solid #aaa;padding:4px 0">{r.date} · {r.label}{mark}
            <div style="display:flex;gap:12px"><img src="{r.iiif_link}" style="max-width:45%;max-height:260px">
            <div style="font-size:12px">{html.escape(r.text)}</div></div></div>""")
    display(HTML("".join(parts)))

In [ ]:
show_cluster(seed=0)

In [ ]:
show_cluster(seed=1)

Official notices are the riskiest: templates that differ only in the place name.

In [ ]:
show_cluster("dup_cluster_size BETWEEN 2 AND 6 AND ad_id IN (SELECT ad_id FROM ads WHERE text LIKE '%Staats%' OR text LIKE '%k. k.%')", seed=2)

Long runs: ads printed for months.

In [ ]:
show_cluster("dup_cluster_size >= 15", seed=3)